## Minimal example of calculating alpha_npq from fluorescence and PAR

In [ ]:
import xarray as xr
import numpy as np

import matplotlib.pyplot as plt
import argopy
from argopy import ArgoFloat

import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, '/home/jovyan/team-iron-stress/scripts')


In [ ]:
# import npq packages
import fit_npq
import npqc
from mld_calculator import mld_calc
from par15 import par15

In [ ]:
# fetch a single Sprof file 
WMOs=['1902593','5907054','2903884','5906765','1902371','2903938'] #pick float number
with argopy.set_options(mode='expert'):
    ds = ArgoFloat(WMOs[0]).open_dataset('Sprof')


In [ ]:
# Ken's method for determining NPQ compares these two variables -- not needed here
#fig, ax = plt.subplots()
#plt.plot(ds['CHLA_ADJUSTED'].T, ds['CHLA_FLUORESCENCE_ADJUSTED'].T, '.');

In [ ]:
ds = par15(ds).rename({'par_15_pressure': 'par_depth'})
ds = mld_calc(ds)

ds = ds#.isel(N_PROF=slice(30))
ds = ds.dropna(dim='N_PROF', subset=['par_depth', 'ml_depth'])
ds = npqc.calculate_NPQ_fluo(ds)

In [ ]:
fig, ax = plt.subplots()
plt.plot(ds['DOWNWELLING_PAR'].T, ds['PRES_ADJUSTED'].T, '.');
ax.set_ylabel('Depth [dbar]')
ax.set_xlabel('Measured PAR')
ax.set_ylim([200, 0]);

fig, ax = plt.subplots()
plt.plot(ds['fluo_smooth'].T, ds['PRES_ADJUSTED'].T, '.');
ax.set_ylabel('Depth [dbar]')
ax.set_xlabel('C')
ax.set_ylim([200, 0]);


In [ ]:
ds = fit_npq.fit_alpha_npq(ds)

In [ ]:
fig, ax = plt.subplots()
plt.plot(ds['par'].T, ds['PRES_ADJUSTED'].T, '.');
ax.set_ylabel('Depth [dbar]')
ax.set_xlabel('Adjusted PAR')
ax.set_ylim([200, 0]);

fig, ax = plt.subplots()
plt.plot(ds['npq'].T, ds['PRES_ADJUSTED'].T, '.');
ax.set_ylabel('Depth [dbar]')
ax.set_xlabel('NPQ')
ax.set_ylim([200, 0]);


In [ ]:
fig, ax = plt.subplots()
plt.plot(ds['par'].T, ds['npq'].T,'.');
ax.set_ylabel('NPQ')
ax.set_xlabel('PAR')

In [ ]:
alpha_NPQ_QC = ds['alpha_NPQ'].where((ds['NPQ_max'] < 19.99999) & (ds['alpha_NPQ'].values < 0.09999))
ds['alpha_NPQ_QC'] = alpha_NPQ_QC

In [ ]:
fig, ax = plt.subplots()
ds['NPQ_max'].plot.line('.-', label='NPQ_max')
ax.set_title('NPQ_max')
fig, ax = plt.subplots()
alpha_NPQ_QC.plot.line('.-', label='alpha_NPQ');
ax.set_title('alpha_NPQ')